# Лабораторная работа 2. Сравнение инструментов обработки данных

**Вариант 6.** Датасет — [Hospital Readmission Prediction](https://www.kaggle.com/datasets/siddharth0935/hospital-readmission-predictionsynthetic-dataset), как в ЛР-1. Работа выполнена на CPU, трек A.



Выполняйте из папки `lab_2`. Полный повтор экспериментов занимает время и создаёт около 1,3 ГБ временных данных. Готовые измерения находятся в `results/`.


In [ ]:
# Повторить все этапы последовательно; используются те же скрипты, что в отчёте.
import subprocess, sys
subprocess.run([sys.executable, "run.py"], check=True)


## 1. Подготовка и задача

Задача: группировка по `gender`, `discharge_destination`, `readmitted_30_days`; число записей, сумма и средняя длительность пребывания, средний BMI. Это описательная оценка нагрузки по группам. Синтетический набор не доказывает клинические закономерности.

В исходнике 30000 строк. Удалено дубликатов: 0, записей с критическими пропусками: 0; очищено 30000 строк. Все 5 проверок Great Expectations прошли: ID заполнен и уникален, длительность 1–10, BMI 0–100, исход `Yes/No`. Проверка BMI задаёт технический контракт набора, а не медицинский диагноз.

| Набор | Строки | CSV, МБ | Parquet, МБ |
|---|---:|---:|---:|
| full | 30 000 | 1,896 | 0,379 |
| 100mb | 1 542 426 | 100,000 | 19,211 |
| 1gb | 15 192 583 | 1 000,000 | 189,139 |

Исходник меньше 100 МБ, поэтому большие наборы получены повторением очищенных записей с уникальными тестовыми ID. Это нагрузочные копии, не новые наблюдения. Размеры 100 МБ/1 ГБ относятся к CSV; Parquet содержит те же строки. Данные генерируются в `tmp/`, в Git хранятся только код и результаты.

Окружение: Python 3.12, pandas 3.1.0, Dask 2026.8.0, Spark 3.5.7, Polars 2.1.0, DuckDB 2.0.0; Windows, 8 логических CPU, 15,85 ГиБ RAM. Spark: `local[4]`, heap 2 ГиБ, исходно 200 shuffle-партиций, AQE выключен.

## 2. Сравнение производительности

Для всех движков одинаковые файлы, составной ключ и агрегаты. Импорт и запуск исключены из времени запроса и сохранены отдельно. Время включает чтение, вычисления и материализацию результата; Spark измеряется после `collect()`, Dask после `compute()`. Каждый набор — отдельный процесс, внутри 3 последовательных прогона; время и CPU — медианы, RSS — максимум. Кэш ОС не очищался.

| Инструмент / CSV | Время, с | RSS, ГиБ | CPU, с | Чтения / записи ОС |
|---|---:|---:|---:|---:|
| pandas / full | 0,045 | 0,083 | 0,05 | 10 / 0 |
| pandas / 100mb | 1,751 | 0,237 | 1,75 | 384 / 0 |
| pandas / 1gb | 17,123 | 1,647 | 18,47 | 3 817 / 0 |
| dask / full | 0,083 | 0,135 | 0,09 | 7 / 0 |
| dask / 100mb | 1,288 | 0,430 | 1,97 | 11 / 0 |
| dask / 1gb | 10,058 | 0,556 | 20,75 | 119 / 0 |
| spark / full | 0,927 | 0,754 | 5,59 | 168 / 38 |
| spark / 100mb | 2,210 | 0,923 | 11,12 | 1 996 / 156 |
| spark / 1gb | 12,179 | 0,973 | 51,12 | 16 091 / 308 |

RSS измерялся каждые 20 мс по процессу и потомкам, включая JVM. CPU — суммарное процессорное время, а не длительность запроса. I/O — счётчики операций ОС: включают служебный ввод-вывод, IPC и работу JVM; mmap и кэш отражаются в них иначе, поэтому это не число физических обращений к диску. Байты I/O и нормированный CPU% сохранены в `results/`.

![Время](assets/time.png)
![Память](assets/memory.png)

На 1 ГБ относительное сокращение времени Dask к pandas: **41,26%** по формуле `(t_pandas − t_dask) / t_pandas`. Отрицательное значение означает замедление. Pandas успешно обработал 1 ГБ: предел отказа в этом опыте не найден, универсальную границу памяти из этих замеров вывести нельзя. Для прямого сравнения Spark дополнительно измерен на тех же 100 МБ и 1 ГБ, а не только на полном исходнике.

## 3. Физический план Spark

Базовый план: `FileScan CSV → partial HashAggregate → Exchange hashpartitioning(200) → final HashAggregate`. Catalyst оставил только 5 нужных столбцов, Tungsten/whole-stage code generation виден по меткам `*(...)`. Полные планы до/после AQE и salting: [results/plans](results/plans).

В первом измеренном baseline на 1 ГБ число выполненных стадий — **2**; записано в shuffle **17,03 КиБ**, прочитано **17,03 КиБ**. Данные получены из Spark status REST API после выполнения запроса. Вход — сумма счётчиков Spark `inputBytes` по задачам, а не размер файла в каталоге.

| Stage | Задач | Вход, МБ | Запись shuffle, КиБ | Сумма executorRunTime, мс |
|---|---:|---:|---:|---:|
| 2 | 8 | 1 000,459 | 17,03 | 46 527 |
| 3 | 200 | 0,000 | 0,00 | 1 253 |

На завершающей стадии непустых задач **18 из 200**; их медианное/max время executor — **20,0 / 30,0 мс**. Пустые задачи исключены из этой статистики, но участвуют в накладных расходах планирования.

Первая стадия читает файл и предварительно агрегирует; следующая объединяет частичные суммы. Сумма времени задач не равна времени стадии из-за параллелизма. Основные гипотезы: уменьшить накладные расходы 200 задач для 18 групп и ускорить чтение форматом Parquet. Для этой агрегации map-side combine резко уменьшает shuffle относительно размера входного CSV.

## 4. Skew и salting

В исходнике 18 составных ключей. Самый частый: `Other / Home / No` — 21,10% строк. Коэффициент вариации частот `std/mean` = **1,265**; max/mean = **3,798**.

![Ключи](assets/skew.png)

Реализована соль `patient_id % 4`: сначала группировка по ключу с солью, затем объединение сумм и счётчиков; средние считаются через сумму/число, не как среднее средних. На компактных частотах исходника оценена загрузка 8 hash-партиций: CV **0,652 → 0,622**. Это оценка распределения исходных строк, а не измеренный shuffle Spark.

На 1 ГБ salting по отношению к варианту `shuffle8` дал **0,89×** (`t_shuffle8/t_salt4`). Значение меньше 1 означает ухудшение. Перекос частот сам по себе не доказывает узкое место reducer: частичная агрегация уже сжимает группы, а соль добавляет вторую перегруппировку. Фактические task-метрики и объём shuffle сохранены для каждого прогона.

## 5. Оптимизации Spark

По 3 прогона на 1 ГБ, с разогревом JVM вне замеров и сменой порядка вариантов. Таблица сравнивает все варианты с CSV/200/AQE-off; `combined` включает Parquet, 8 партиций и AQE. Для `salt4` дополнительно рассчитано сравнение с `shuffle8` в разделе 4.

| Вариант | Время, с | baseline / время | Запись shuffle, КиБ | Стадий, 1-й прогон |
|---|---:|---:|---:|---:|
| baseline | 13,003 | 1,00× | 17,03 | 2 |
| shuffle8 | 12,018 | 1,08× | 12,13 | 2 |
| repartition8 | 14,996 | 0,87× | 140 983,38 | 2 |
| aqe | 13,254 | 0,98× | 17,03 | 2 |
| parquet | 2,057 | 6,32× | 8,54 | 3 |
| combined | 1,564 | 8,32× | 6,08 | 3 |
| salt4 | 13,473 | 0,97× | 35,99 | 3 |

`shuffle8`: только 200 → 8; `repartition8`: дополнительное распределение сырых строк по ключу; `aqe`: только AQE; `parquet`: только формат. Для Parquet API дополнительно учитывает служебную стадию чтения метаданных, поэтому 3 стадии не означают 2 shuffle. Вариант с минимальной медианой среди проверенных: **combined**, **1,564 с**. Это лучший из измеренных вариантов, не полный поиск всех настроек. Время предварительного преобразования CSV → Parquet не включено: выигрыш окупается при повторных чтениях.

![Затраты и выгода](assets/optimization.png)

## 6. CPU-трек A: pandas, Polars, DuckDB

Тот же запрос и те же CSV, по 3 прогона. Polars использует ленивое streaming-выполнение и 4 потока, DuckDB — SQL и 4 потока; pandas — C-парсер и группировку DataFrame.

| Инструмент / CSV | Время, с | RSS, ГиБ | CPU, с | Чтения / записи ОС |
|---|---:|---:|---:|---:|
| pandas / 100mb | 1,751 | 0,237 | 1,75 | 384 / 0 |
| pandas / 1gb | 17,123 | 1,647 | 18,47 | 3 817 / 0 |
| polars / 100mb | 0,226 | 0,326 | 0,75 | 4 / 0 |
| polars / 1gb | 1,865 | 0,656 | 7,12 | 31 / 0 |
| duckdb / 100mb | 0,518 | 0,204 | 1,92 | 4 / 0 |
| duckdb / 1gb | 4,632 | 0,291 | 19,55 | 32 / 0 |

На 1 ГБ сокращение времени против pandas: Polars **89,11%**, DuckDB **72,95%**. Возможные причины различий — параллельная обработка, векторизация и продвижение проекции. Отдельного GPU-замера нет: выполнен обязательный CPU-трек. На коротких запросах накладные расходы GPU могут перекрыть выигрыш; утверждать GPU-ускорение по CPU-таблице нельзя. Команды и таблица трека: [gpu/README.md](gpu/README.md).

## 7. Выбор инструмента и стоимость

| Условие | Выбор | Компромисс |
|---|---|---|
| Исходные 30 000 строк, простые проверки | pandas | Ограничение памятью одного процесса |
| Аналитический SQL на одном узле | DuckDB | Не полноценная распределённая платформа |
| Большие локальные таблицы и ленивые вычисления | Polars | Другой API и модель выполнения |
| Существующий pandas-пайплайн, нужно масштабирование | Dask | Планирование задач и контроль размера партиций |
| Данных больше ресурсов узла, сложные распределённые ETL | Spark | JVM, shuffle и эксплуатация кластера |

Учебная модель стоимости: одинаковая машина 8 CPU/16 ГиБ за **20 ₽/час**; один холодный запуск стоит `(startup + median_query) × 20 / 3600`. Разработка/поддержка ниже — оценочные трудозатраты, не фактически засечённое время; при 1 000 ₽/час годовая трудовая часть = `(разработка + 12 × поддержка) × 1000`. Хранение и сеть в эту частичную модель не входят.

| Инструмент | Вычисления 1 ГБ, ₽/запуск | Разработка, ч | Поддержка, ч/месяц |
|---|---:|---:|---:|
| pandas | 0,0994 | 1,0 | 0,5 |
| dask | 0,0663 | 2,0 | 1,0 |
| spark | 0,1428 | 4,0 | 2,0 |
| polars | 0,0173 | 1,5 | 0,5 |
| duckdb | 0,0318 | 1,5 | 0,5 |

Для текущего медицинского набора достаточно pandas; Polars/DuckDB стоит рассмотреть при росте локальных выгрузок. Spark обоснован при требованиях распределённого ETL, а не только размером CSV. Переход требует контроля схемы, корректности групп и сроков обработки. Данные без дат не позволяют обещать SLA обученного временного прогноза.

**Итог:** результаты всех пяти движков проверены по всем ключам, числу записей, суммам и средним; целочисленные агрегаты совпадают точно, для BMI разрешена погрешность вычислений с плавающей точкой. Измерены реальные стадии, shuffle, CPU и I/O; отрицательный эффект оптимизации сохраняется как результат.

Источники: предоставленные задание ЛР-2 и лекция 2; [Spark Monitoring](https://spark.apache.org/docs/3.5.7/monitoring.html), [Spark SQL tuning](https://spark.apache.org/docs/3.5.7/sql-performance-tuning.html), [Polars](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect.html), [DuckDB](https://duckdb.org/docs/stable/clients/python/overview.html). МБ = 10⁶ байт, ГБ = 10⁹, ГиБ = 2³⁰. Локальные замеры не заменяют тестирование в распределённом кластере.
